In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import scipy.sparse as sp
import anndata as ad
import gseapy as gp
from copy import deepcopy
import scanpy.external as sce
import pickle
from scipy import sparse


# sc.logging.print_version()
sc.settings.verbosity = 3
sc.set_figure_params(figsize=(5,5))

import sys
from pathlib import Path

_start = Path.cwd().resolve()
_repo_root = next((p for p in [_start, *_start.parents] if (p / "chronopack").exists()), _start)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import chronopack as cp
from chronopack.tl import _gene_dynamics as gd
from chronopack.preprocessing import filter_low_count_genes, filter_technical_genes


In [ ]:
# using CPM @ 1e6 scale factor
adata_hvg = sc.read_h5ad('../output/adata_hvg.h5ad')
adata = sc.read_h5ad('../output/adata_full.h5ad')

# Prepare input files for TradeSeq

In [ ]:
mtx_agg, cpm_ref, logcpm_ref = gd.load_negative_control_logcpm('../../thp1-k562-lps.hg38.rep/intmd/k562_neg-ctrl_mtx-agg.pkl')


In [ ]:
cpm, logcpm = gd.temporal_pseudobulk_from_anndata(adata)


In [ ]:
shared_genes = list(set(adata.var.index) & set(logcpm_ref.columns))

cpm_ref = cpm_ref.loc[:, shared_genes]
logcpm_ref = logcpm_ref.loc[:, shared_genes]

cpm = cpm.loc[:, shared_genes]
logcpm = logcpm.loc[:, shared_genes]

adata.shape

In [ ]:
corr = gd.compute_smooth_correlations(
    logcpm,
    logcpm_ref,
    method='spline',
    method_kwargs=dict(s=None, k=3),
    local_r_threshold=0.8,
    local_fraction_threshold=0.5,
)


In [ ]:
corr = gd.refilter_tag_bias_candidates(
    corr,
    local_r_threshold=0.7,
    local_fraction_threshold=0.25,
    use_residual_only=True,
)

print("Tag-bias candidates:", int(corr["tag_bias_candidate"].sum()))

corr[
    [
        "pearson_r_residual",
        "residual_explained_total_fraction",
        "pearson_r_diff",
        "diff_residual_explained_fraction",
        "tag_bias_candidate_residual",
        "tag_bias_candidate_diff",
        "tag_bias_candidate",
    ]
].head(20)


In [ ]:
excl_genes = gd.tag_bias_exclusion_genes(corr)

In [ ]:
'HSPA1B' in excl_genes

In [ ]:
adata_hvg = adata.copy()
adata_hvg

In [ ]:
adata_hvg = adata.copy()
adata_hvg = filter_technical_genes(adata_hvg, exclude_genes=excl_genes)
adata_hvg = filter_low_count_genes(adata_hvg, min_total_counts=10, counts_layer="counts")
adata_hvg


In [ ]:
print('PLAU' in set(adata_hvg.var.index)) # similar shape
print('CD83' in set(adata_hvg.var.index)) # similar shape
print('EGR1' in set(adata_hvg.var.index)) # similar shape
print('IER3' in set(adata_hvg.var.index)) # similar shape

In [ ]:
# TPs
print('REL' in set(adata_hvg.var.index)) # real, should be included
print('ALAS1' in set(adata_hvg.var.index)) # also real, should be included
print('SLC40A1' in set(adata_hvg.var.index)) # similar shape but a lot noisier
print('TRAF3' in set(adata_hvg.var.index)) # similar shape

# TNs
print('FTH1' in set(adata_hvg.var.index)) # should have high correlation
print('RELA' in set(adata_hvg.var.index)) # similar shape

In [ ]:
print(f"Genes retained after technical, time-tag-bias, and low-count filtering: {adata_hvg.n_vars}")
print("Low-count threshold applied before writing tradeSeq inputs: total counts >= 10")


In [ ]:
pd.Series(list(excl_genes)).to_csv('exclude_time-tag-bias_genes.csv', index=False, )
len(list(excl_genes))

In [ ]:
adata_hvg.write_h5ad('../output/adata_prefiltered.1e4-variant.fixed.h5ad')
adata_full = adata.copy()

In [ ]:
cp.tl.fit_gene_dynamics(
    adata_hvg,
    output_dir='../intmd/sc-K562-1e6-to-tradeSeq.fixed',
    full_adata=adata,
    prepare_inputs=True,
    run_backend=False,
)
